# 06 - Decision engine: when to roll, level or save

**Question:** given my round, level, XP, gold, HP, streak and the unit I am chasing, what should I do now?

**How the pieces fit**
- **Gold cost of an upgrade** comes from the Markov chain of notebook 04, re-created here so this notebook stands alone.
- **Rules** (XP table, income, interest, streaks, damage) are Set 18 values from Riot's patch notes and OP.GG, patch 18.4. **Check them in-game** (see section 1).
- **HP danger** comes from a small lobby simulator, calibrated so that it reproduces the survival curve measured in notebook 05.
- **Placement value** of an upgrade comes from notebook 03 (correlational, so treat it as an estimate).

**What this is and is not.** It is a transparent decision-support calculator with adjustable assumptions. It does not prove that a given action improves placement: the match data has no per-round state, so no model can learn that from it.

Needs: `data/units.csv`, and optionally `hazard_by_round.csv` (notebook 05) and `value_table.csv` (notebook 03).

## 1. Rules and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from math import comb, ceil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- Set 18 rules (patch 18.4). VERIFY IN-GAME. ----------------
# XP needed to go from level-1 to level. Level 8 follows Riot's 18.2 notes (56); OP.GG still lists 60.
XP_TO_REACH = {3: 2, 4: 6, 5: 10, 6: 20, 7: 36, 8: 60, 9: 68, 10: 68}
XP_PER_ROUND = 2          # free XP each round
GOLD_PER_XP = 1           # 4 XP costs 4 gold
REROLL_COST = 2
SLOTS = 5
COPIES_FOR_STAR = {2: 3, 3: 9}
START_HP = 100            # assumption, check your client
# player damage = damage per stage + one per surviving enemy unit
STAGE_DMG = {1: 0, 2: 2, 3: 5, 4: 8, 5: 10, 6: 12, 7: 17}
SURV_MEAN = {1: 0, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7}    # rough enemy survivors per stage, scaled by calibration
# streak bonus as listed by OP.GG for Set 18 (older sets differed, so double-check)
STREAK_BONUS = {0: 0, 1: 0, 2: 1, 3: 1, 4: 1, 5: 2}

ODDS = pd.DataFrame(
    [[100, 0, 0, 0, 0], [100, 0, 0, 0, 0], [75, 25, 0, 0, 0], [55, 30, 15, 0, 0],
     [45, 33, 20, 2, 0], [30, 40, 25, 5, 0], [16, 30, 43, 10, 1], [15, 20, 32, 30, 3],
     [10, 17, 25, 33, 15], [5, 10, 20, 40, 25], [1, 2, 12, 50, 35]],
    index=range(1, 12), columns=range(1, 6)) / 100
POOL = pd.DataFrame({"champions": [14, 13, 14, 14, 10], "copies_each": [30, 25, 18, 10, 9]}, index=range(1, 6))
POOL["tier_total"] = POOL["champions"] * POOL["copies_each"]
OTHERS = 7


# ---------------- round bookkeeping ----------------
# last_round in the match data is a global counter: stage 1 has 4 rounds, every later stage has 7.
# The six zero-death rounds seen in notebook 05 (22, 25, 29, 32, 36, 39) are exactly rounds 4 and 7 of stages 4-6.
def stage_round(r):
    if r <= 4:
        return 1, r
    return 2 + (r - 5) // 7, (r - 5) % 7 + 1

def round_number(stage, rnd):
    return rnd if stage == 1 else 4 + (stage - 2) * 7 + rnd

def is_pvp(r):
    s, k = stage_round(r)
    return s >= 2 and k in (1, 2, 3, 5, 6)      # x-4 and x-7 are carousel / PvE rounds

def base_income(r):
    return 0 if r <= 1 else 2 if r in (2, 3) else 3 if r == 4 else 4 if r == 5 else 5

def interest(gold):
    return min(int(gold) // 10, 5)

def streak_bonus(streak):
    n = abs(int(streak))
    return 3 if n >= 6 else STREAK_BONUS[n]

def next_round_income(r_now, gold, streak, win_rate):
    r = r_now + 1
    win = win_rate if is_pvp(r) else 0.0
    return base_income(r) + interest(gold) + win + streak_bonus(streak)

def xp_gold_to_level(level, xp, target_level):
    '''Gold to buy XP from (level, xp) up to target_level, in 4-XP steps.'''
    if target_level <= level:
        return 0
    need = sum(XP_TO_REACH[l] for l in range(level + 1, target_level + 1)) - xp
    return int(ceil(max(need, 0) / 4) * 4 * GOLD_PER_XP)

print("Check: stage-round 4-4 is round", round_number(4, 4), "-> back:", stage_round(22))
print("PvP rounds in stage 4:", [stage_round(r) for r in range(19, 26) if is_pvp(r)])

## 2. Pool pressure from your match data
Same estimate as notebook 04: how many copies players hold of a unit they field, and how much of each cost tier the other 7 players occupy.

In [ ]:
DATA = next(p for p in [Path("data"), Path("../data"), Path("D:/VS Code/data")] if (p / "units.csv").exists())
units = pd.read_csv(DATA / "units.csv")
units["cost"] = units["rarity"] + 1
units["copies"] = units["tier"].map({1: 1, 2: 3, 3: 9}).fillna(9)
n_boards = units["board_id"].nunique()
held_when_played = units.groupby("cost")["copies"].mean()
per_board = units.groupby("cost")["copies"].sum() / n_boards
BG = (OTHERS * per_board / POOL["tier_total"]).clip(upper=0.9).fillna(0)
pd.DataFrame({"copies_held_per_fielded_unit": held_when_played.round(2), "background_depletion": BG.round(3)})

## 3. Gold cost of an upgrade (Markov chain from notebook 04)

In [ ]:
def slot_probs(level, cost, n_need, contesters=0, bg_scale=1.0):
    total = POOL.loc[cost, "copies_each"]
    taken = contesters * float(held_when_played.get(cost, 3.0))
    if total - taken < n_need:
        return None
    other_left = (POOL.loc[cost, "tier_total"] - total) * (1 - bg_scale * float(BG.get(cost, 0.0)))
    s = np.arange(n_need)
    target_left = total - taken - s
    return np.clip(ODDS.loc[level, cost] * target_left / (target_left + other_left), 0, 1)

def simulate_gold(level, cost, n_need, owned=0, contesters=0, n_sim=8000, seed=0):
    '''Gold spent (rerolls + copies) to go from `owned` to `n_need` copies. First shop view is free.'''
    pr = slot_probs(level, cost, n_need, contesters)
    if pr is None or (pr[owned:] <= 0).any():
        return np.full(n_sim, np.inf)
    rng = np.random.default_rng(seed)
    p_full = np.append(pr, 0.0)
    s = np.full(n_sim, owned)
    shops = np.zeros(n_sim, dtype=int)
    alive = s < n_need
    while alive.any() and shops.max() < 20000:
        hits = rng.binomial(SLOTS, p_full[s])
        s = np.where(alive, np.minimum(n_need, s + hits), s)
        shops += alive
        alive = s < n_need
    return REROLL_COST * np.maximum(shops - 1, 0) + cost * (n_need - owned)

demo = simulate_gold(8, 3, 3, owned=1, contesters=1)
print("Level 8, 3-cost to 2-star from 1 copy, 1 contester: mean", round(demo.mean()), "gold, 90th pct", round(np.percentile(demo, 90)))

## 4. HP model: a small lobby simulator, calibrated to the real survival curve
Eight players, random pairings each PvP round, the loser takes `stage damage + surviving enemy units`. Players differ in strength, and a logistic curve with slope `gamma` turns strength gaps into win probabilities.
Two numbers are fitted so that simulated deaths match the Kaplan-Meier curve of notebook 05: `gamma` (how much skill decides fights) and `surv_scale` (how many enemy units survive a typical loss). Everything else is rule-based.

**Limit:** `START_HP` (assumed 100) and `surv_scale` both change how fast HP drains, so the data cannot tell them apart. The fit shows the HP model is consistent with real timing, not that each number is individually right. If the best `gamma` or `surv_scale` sits on the edge of the grid, widen the grid.

In [ ]:
PVP_ROUNDS = [r for r in range(5, 60) if is_pvp(r)]

def simulate_lobbies(n_lobbies, gamma, surv_scale, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_lobbies):
        skill = rng.normal(0, 1, 8)
        hp = np.full(8, float(START_HP))
        alive = list(range(8))
        last = PVP_ROUNDS[0]
        for r in PVP_ROUNDS:
            if len(alive) <= 1:
                break
            last = r
            st = min(stage_round(r)[0], 7)
            order = rng.permutation(alive)
            for k in range(0, len(order) - 1, 2):
                a, b = order[k], order[k + 1]
                pa = 1.0 / (1.0 + np.exp(-gamma * (skill[a] - skill[b])))
                loser = b if rng.random() < pa else a
                hp[loser] -= STAGE_DMG[st] + max(1, rng.poisson(surv_scale * SURV_MEAN[st]))
            for p in [p for p in alive if hp[p] <= 0]:
                out.append((r, 1))
            alive = [p for p in alive if hp[p] > 0]
        for _ in alive:
            out.append((last, 0))          # survivors are censored
    return pd.DataFrame(out, columns=["last_round", "event"])

def km_table(df, time="last_round", event="event"):
    t = df.groupby(time).agg(n_events=(event, "sum"), n_total=(event, "size")).sort_index()
    t["at_risk"] = t["n_total"][::-1].cumsum()[::-1]
    t["hazard"] = t["n_events"] / t["at_risk"]
    t["survival"] = (1 - t["hazard"]).cumprod()
    return t

def survival_at(t, rounds):
    return t["survival"].reindex(range(int(min(rounds)) - 1, int(max(rounds)) + 1)).ffill().fillna(1.0).reindex(rounds)

obs_path = DATA / "hazard_by_round.csv"
OBS = pd.read_csv(obs_path, index_col="last_round")["survival"] if obs_path.exists() else None
CHECK = [24, 26, 27, 28, 30, 31, 33, 34, 35, 37, 38, 40]
print("Observed survival curve loaded:", OBS is not None)

In [ ]:
rows = []
if OBS is not None:
    obs_vals = OBS.reindex(range(int(OBS.index.min()) - 1, 46)).ffill().fillna(1.0).reindex(CHECK)
    for gamma in [0.0, 0.25, 0.5, 1.0, 1.5]:
        for sc in [0.8, 1.0, 1.2, 1.4, 1.6, 2.0]:
            sim = km_table(simulate_lobbies(300, gamma, sc, seed=1))
            err = float(((survival_at(sim, CHECK) - obs_vals) ** 2).mean())
            rows.append(dict(gamma=gamma, surv_scale=sc, mse=round(err, 5)))
    grid = pd.DataFrame(rows).sort_values("mse").reset_index(drop=True)
    display(grid.head(6))
    GAMMA, SURV_SCALE = float(grid.loc[0, "gamma"]), float(grid.loc[0, "surv_scale"])
else:
    GAMMA, SURV_SCALE = 1.0, 1.0
    print("No hazard_by_round.csv found, using defaults gamma=1, surv_scale=1 (uncalibrated).")
print("Using gamma =", GAMMA, " surv_scale =", SURV_SCALE)

In [ ]:
sim = km_table(simulate_lobbies(1500, GAMMA, SURV_SCALE, seed=7))
fig, ax = plt.subplots(figsize=(8, 5))
ax.step(sim.index, sim["survival"], where="post", label="simulated lobby")
if OBS is not None:
    ax.step(OBS.index, OBS.values, where="post", label="observed (notebook 05)")
ax.set_xlabel("round"); ax.set_ylabel("share still alive"); ax.legend()
ax.set_title("Does the rule-based HP model reproduce real elimination timing?")
plt.show()

In [ ]:
def survival_odds(hp, round_now, win_rate, horizon_pvp=3, n=6000, seed=3):
    '''Chance of still being alive after the next `horizon_pvp` PvP rounds, given a flat per-fight win rate.'''
    rng = np.random.default_rng(seed)
    nxt = [r for r in PVP_ROUNDS if r > round_now][:horizon_pvp]
    cur = np.full(n, float(hp))
    for r in nxt:
        st = min(stage_round(r)[0], 7)
        lose = rng.random(n) > win_rate
        dmg = STAGE_DMG[st] + np.maximum(1, rng.poisson(SURV_SCALE * SURV_MEAN[st], n))
        cur = cur - lose * dmg
    return float((cur > 0).mean())

print("P(alive after 3 PvP rounds) at 4-1 with 70 HP, 50% win rate:", round(survival_odds(70, round_number(4, 1), 0.5), 3))
print("Same with 25 HP and a 40% win rate:", round(survival_odds(25, round_number(4, 1), 0.4), 3))

## 5. Placement value of an upgrade (from notebook 03)
Read from `value_table.csv` if present. The first 3-star unit was worth roughly 0.44 placements and a 2-star roughly 0.14 per unit on average, from the notebook 03 SHAP results.
These are correlational and, for 3-stars, mostly a reroll-comp effect, so they only scale the final column.

In [ ]:
GAIN = {2: 0.14, 3: 0.44}
vt_path = DATA / "value_table.csv"
if vt_path.exists():
    vt = pd.read_csv(vt_path)
    t3 = vt[vt["feature"] == "n_3star"].set_index("value")["mean"]
    if 0.0 in t3.index and 1.0 in t3.index:
        GAIN[3] = round(float(t3[0.0] - t3[1.0]), 2)
print("Placement gain used (positive = better):", GAIN)

## 6. The advisor
For a target unit it compares: rolling at the current level, levelling up first (1 to 3 levels), and waiting one round. Each plan is scored in gold:
`expected gold + interest given up over the next rounds`. Waiting is costed the same way, using next round's gold, so its gold cost is similar: its real price is one more round with the weaker board, which gold cannot show. For that reason plans you can already afford are preferred over waiting. If HP is critical it picks the plan most likely to be affordable right now instead.

In [ ]:
def advise(round_now, level, xp, gold, hp, streak, target_cost, target_star=2, owned=0,
           contesters=0, win_rate=0.5, horizon_rounds=2, n_sim=8000):
    n_need = COPIES_FOR_STAR[target_star]
    inc = next_round_income(round_now, gold, streak, win_rate)
    rows = []
    for L2 in range(level, min(level + 3, 10) + 1):
        xg = xp_gold_to_level(level, xp, L2)
        sim = simulate_gold(L2, target_cost, n_need, owned, contesters, n_sim)
        exp_g = float(sim.mean())
        spend = xg + exp_g
        int_cost = horizon_rounds * max(interest(gold) - interest(max(gold - spend, 0)), 0) if np.isfinite(spend) else np.inf
        rows.append(dict(plan="roll at level %d" % L2 if L2 == level else "level to %d, then roll" % L2,
                         xp_gold=xg, exp_roll_gold=exp_g, p90_roll_gold=float(np.percentile(sim, 90)),
                         p_afford_now=float((sim <= gold - xg).mean()) if gold - xg >= 0 else 0.0,
                         p_afford_next_round=float((sim <= gold + inc - xg).mean()) if gold + inc - xg >= 0 else 0.0,
                         interest_cost=int_cost, total_cost=spend + int_cost))
    sim0 = simulate_gold(level, target_cost, n_need, owned, contesters, n_sim)
    gold_next = gold + inc
    spend_w = float(sim0.mean())
    int_w = horizon_rounds * max(interest(gold_next) - interest(max(gold_next - spend_w, 0)), 0) if np.isfinite(spend_w) else np.inf
    rows.append(dict(plan="wait one round (save)", xp_gold=0, exp_roll_gold=spend_w,
                     p90_roll_gold=float(np.percentile(sim0, 90)), p_afford_now=0.0,
                     p_afford_next_round=float((sim0 <= gold_next).mean()),
                     interest_cost=int_w, total_cost=spend_w + int_w))
    df = pd.DataFrame(rows)

    alive = survival_odds(hp, round_now, win_rate, horizon_pvp=3)
    critical = alive < 0.85
    cand = df[np.isfinite(df["total_cost"])]
    if len(cand) == 0:
        msg = "No affordable path to this unit at these levels. Consider a different target."
    elif critical:
        best = cand.sort_values(["p_afford_now", "total_cost"], ascending=[False, True]).iloc[0]
        msg = ("HP is critical (%.0f%% to survive 3 more PvP rounds), so favour power now: '%s' "
               "(%.0f%% chance you can afford it this round)." % (100 * alive, best["plan"], 100 * best["p_afford_now"]))
    else:
        ok = cand[cand["p_afford_now"] >= 0.5]
        if len(ok):
            best = ok.sort_values("total_cost").iloc[0]
            msg = "Cheapest affordable plan: '%s' (about %.0f gold all-in, %.0f%% to afford it now)." % (
                best["plan"], best["total_cost"], 100 * best["p_afford_now"])
        else:
            best = df[df["plan"].str.startswith("wait")].iloc[0]
            msg = "You probably cannot afford the upgrade this round: save (%.0f%% to afford it next round)." % (
                100 * best["p_afford_next_round"])
    gain_per_100 = GAIN[target_star] / best["total_cost"] * 100 if np.isfinite(best["total_cost"]) and best["total_cost"] > 0 else np.nan
    out = df.copy()
    for c in ["exp_roll_gold", "p90_roll_gold", "interest_cost", "total_cost"]:
        out[c] = out[c].round(0)
    out[["p_afford_now", "p_afford_next_round"]] = out[["p_afford_now", "p_afford_next_round"]].round(2)
    s, k = stage_round(round_now)
    print("Round %d-%d | level %d | gold %d | HP %d | streak %+d | target: %d-cost to %d-star (own %d, %d contesters)"
          % (s, k, level, gold, hp, streak, target_cost, target_star, owned, contesters))
    print(msg)
    print("Rough placement value of this upgrade per 100 gold: %.2f  (correlational estimate)" % gain_per_100)
    return out

## 7. Three example situations

In [ ]:
# Mid-game, healthy: 4-1, level 7, chasing a 3-cost 2-star with one contester
advise(round_number(4, 1), level=7, xp=10, gold=52, hp=70, streak=2, target_cost=3, target_star=2, owned=1, contesters=1)

In [ ]:
# Same target, but HP is low and you are losing
advise(round_number(5, 2), level=8, xp=20, gold=38, hp=21, streak=-3, target_cost=3, target_star=2, owned=2, contesters=1, win_rate=0.4)

In [ ]:
# Expensive target: a 4-cost 2-star at level 7 with 60 gold
advise(round_number(4, 5), level=7, xp=30, gold=60, hp=65, streak=0, target_cost=4, target_star=2, owned=1, contesters=0)

## How to use it and what to state in the write-up
Edit the arguments of `advise(...)` to match your own game: round (`round_number(stage, round)`), level, XP, gold, HP, streak, the cost of the unit you chase, the star you want, copies owned, and how many other players want the same unit.

**Verified against your data:** the round layout (zero-death rounds fall exactly on rounds 4 and 7 of stages 4 to 6) and the HP model (calibrated to the real survival curve).

**Taken from outside sources, check in-game:** XP table, income, interest, streak bonuses, damage, start HP, shop odds.

**Assumptions, not data:** the value of an upgrade in placements; contesters hold the average number of copies seen on final boards; a flat win rate when you pass one; no augments, items or board strength inside the advisor.

**Cannot be learned from this data:** whether slamming items or rolling down early saves HP versus holding gold. That needs round-by-round replays, which the Riot API does not provide.